In [17]:
import re
import unicodedata
import pandas as pd

In [18]:
def strip_accents(s: str) -> str:
    s = "" if s is None else str(s)
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))


def norm_text(s: str) -> str:
    s = str(s)
    s = s.replace("\n", " ").replace("\r", " ")
    s = s.replace("“", '"').replace("”", '"').replace("’", "'")
    s = re.sub(r"\s+", " ", s).strip()
    return s


def norm_key(s: str) -> str:
    s = norm_text(s).lower()
    s = strip_accents(s)
    s = s.replace("–", "-").replace("—", "-")
    s = re.sub(r"\s+", " ", s).strip()
    return s


def prep_question_for_match(s: str) -> str:
    s = norm_key(s)
    # remove Likert explanations like (1 = ... 6 = ...)
    s = re.sub(r"\(\s*1\s*=.*?6\s*=.*?\)", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

In [19]:
DEVICE_MAP = {
    "kis eszkozok": "small",
    "szemelyes it eszkozok": "personal_it",
    "nagy haztartasi gepek": "large_household",
    "dispozitive mici": "small",
    "dispozitive it personale": "personal_it",
    "electrocasnice mari": "large_household",
    "small appliances": "small",
    "personal it devices": "personal_it",
    "large household appliances": "large_household",
    "large household appliances ": "large_household",
}

BASE_PATTERNS = [
    (r"^idobelyeg$|^submitted at$", "submitted_at"),
    (r"^respondent$", "respondent"),
    (r"^neme:?$|^sexul:?$|^gender:?$", "gender"),
    (r"^kora:?$|^varsta:?$|^age:?$", "age"),
    (r"^legmagasabb iskolai vegzettsege:?|^cea mai inalta calificare obtinuta:?|^highest level of finished education:?$", "education"),
    (r"^jelenlegi foglalkozasa:?|^ocupatia actuala:?|^current occupation:?$", "occupation"),
    (r"^lakohelye:?|^loc de domiciliu:?|^place of residence:?$", "residence"),
    (r"^haztartasanak havi netto jovedelme.*|^venitul lunar net al gospodariei.*|^your household's monthly net income.*$", "household_income_eur"),
    (r"^haztartasanak merete:?$|^marimea gospodariei:?$|^your household size$", "household_size"),
    (r"^are you from the netherlands.*$", "nl_residency_filter"),

    (r"^1\. .*environment.*|^1\. aggaszt.*|^1\. ma ingrijoreaza.*", "q01_env_concern"),
    (r"^2\. .*important.*decisions.*not harm.*|^2\. fontosnak tartom.*donteseimmel.*|^2\. consider important ca deciziile.*", "q02_no_harm_decisions"),
    (r"^3\. .*follow rules.*community.*|^3\. igyekszem olyan szabalyokat.*|^3\. incerc sa respect reguli.*", "q03_follow_norms"),
    (r"^4\. .*society.*sustainable.*stable.*|^4\. fontosnak tartom.*tarsadalom fenntarthato.*|^4\. consider important ca societatea.*", "q04_sustainable_society"),
    (r"^5\. .*broken.*smartphone.*|^5\. mit lehet tenni.*okostelefonnal.*|^5\. ce se poate face cu un smartphone.*", "q05_broken_phone_actions"),
    (r"^6\. .*e-waste.*|^6\. hallottal mar.*e-hulladek.*|^6\. ai auzit vreodata.*deseuri electronice.*", "q06_heard_e_waste"),
    (r"^7\. .*participated.*programs.*|^7\. vettel mar reszt.*programokon.*|^7\. ai participat vreodata.*", "q07_joined_programs"),
    (r"^8\. .*interested.*similar events.*|^8\. amennyiben az elozo kerdesre nemmel.*|^8\. daca la intrebarea anterioara ai raspuns.*", "q08_would_join_programs"),

    (r"^1\. .*replace.*before.*worn out.*|^1\. milyen gyakran cser.*|^1\. cu ce frecventa inlocuiesti.*", "q09_replace_frequency"),
    (r"^2\. .*purchased.*used or refurbished.*|^2\. vasaroltal mar hasznalt.*|^2\. ai cumparat vreodata.*second-hand.*", "q10_bought_used"),
    (r"^3\. .*main reason.*answered.*yes.*|^3\. ha igen.*elso.*leges indokod.*|^3\. daca da.*motivul principal.*", "q11_used_main_reason"),
    (r"^4\. .*didn.?t buy secondhand.*held you back.*|^4\. ha nem vasaroltal hasznaltan.*|^4\. daca nu ai cumparat second-hand.*", "q12_not_used_barrier"),
    (r"^5\. .*used any of the following services.*|^5\. igenybe vetted mar valaha az alabbi szolgaltatasokat.*|^5\. ai folosit vreodata urmatoarele servicii.*", "q13_used_services"),
    (r"^6\. .*open.*renting or sharing.*instead of owning.*|^6\. mennyire lennel nyitott arra.*|^6\. cat de deschis.*inchiriezi sau sa folosesti in comun.*", "q14_open_to_rent_share"),
    (r"^7\. .*durability or repairability.*important.*|^7\. fontosnak tartod.*tartos vagy javithato.*|^7\. consideri important.*durabile sau usor de reparat.*", "q15_importance_durability"),

    (r"^1\. if an electronic device breaks down.*usually have it repaired.*|^1\. ha egy elektronikai eszkoz meghibasodik.*|^1\. daca un dispozitiv electronic se defecteaza.*", "q16_usually_repair"),
    (r"^2\. ha egy kis eszkozt nem javittatsz meg.*|^if you don't get a small appliance repaired.*|^2\. daca nu repari un dispozitiv mic.*", "q17_small_not_repair_reason"),
    (r"^3\. ha egy szemelyes it eszkozt nem javittatsz meg.*|^if you don't get a personal it device repaired.*|^3\. daca nu repari un dispozitiv it personal.*", "q18_it_not_repair_reason"),
    (r"^4\. ha egy nagy meretu haztartasi eszkozt nem javittatsz meg.*|^if you don't get a large household appliance repaired.*|^4\. daca nu repari un electrocasnic mare.*", "q19_large_not_repair_reason"),
    (r"^5\. mi osztonozne.*javittasd.*|^what would most encourage you.*repaired rather than replaced.*|^5\. ce te-ar motiva cel mai mult sa repari.*", "q20_repair_motivation"),
    (r"fix minor problems yourself|sajat magad megjavitani|repari singur", "q21_fix_minor_yourself"),
    (r"^7\. .*fully repaired.*|^7\. mennyire vagy hajlando.*teljes mertekben megjavittatva.*|^7\. in ce masura esti dispus.*dupa ce a fost complet reparat.*", "q22_willing_after_full_repair"),
    (r"^8\. .*minor issues remained.*|^8\. mennyire vagy hajlando.*kisebb fennmarado hibak.*|^8\. in ce masura esti dispus.*defecte minore.*", "q23_willing_after_imperfect_repair"),

    (r"^1\. when you no longer wish to use a small device.*|^1\. ha egy kis meretu eszkozt mar nem szeretnel hasznalni.*|^1\. daca nu mai vrei sa utilizezi un dispozitiv mic.*", "q24_small_end_of_life"),
    (r"^2\. if you no longer want to use a personal it device.*|^2\. ha egy szemelyes it eszkozt mar nem szeretnel hasznalni.*|^2\. daca nu mai vrei sa utilizezi un dispozitiv it personal.*", "q25_it_end_of_life"),
    (r"^3\. if you no longer want to use a large household appliance.*|^3\. ha egy nagy haztartasi gepet mar nem szeretnel hasznalni.*|^3\. daca nu mai doresti sa folosesti un electrocasnic mare.*", "q26_large_end_of_life"),
    (r"^4\. do you know where the nearest e-waste drop-off.*|^4\. tudod, hogy lakohelyed kozeleben hol talalhato e-hulladek leadohely.*|^4\. stii unde se afla cel mai apropiat punct de predare.*", "q27_knows_dropoff"),
    (r"^5\. what would encourage you to take the device.*|^5\. mi oszt.*ozne arra, hogy megfelelo helyre vidd.*|^5\. ce te-ar determina sa duci dispozitivul.*", "q28_dropoff_motivation"),
    (r"^6\. what.?s stopping you from dropping-off.*|^6\. mi akadalyozza meg, hogy leadd.*|^6\. ce te impiedica sa predai dispozitivele.*", "q29_dropoff_barrier"),
    (r"^7\. would you be willing to do more in the future.*|^7\. a jovoben hajlando lennel-e tobbet tenni.*|^7\. esti dispus.*sa faci mai mult in viitor.*", "q30_willing_future_disposal"),
]

In [20]:
DEVICE_BRACKET_RE = re.compile(r"^(.*)\[(.*?)\]\s*$")


def detect_base_name(col: str) -> str | None:
    k = prep_question_for_match(col)
    for pat, cname in BASE_PATTERNS:
        if re.search(pat, k):
            return cname
    return None


def canonical_col(col: str) -> str:
    raw = norm_text(col)
    raw_for_match = prep_question_for_match(raw)

    m = DEVICE_BRACKET_RE.match(raw_for_match)
    if m:
        base_txt = m.group(1).strip()
        device_txt = norm_key(m.group(2))
        base_name = detect_base_name(base_txt)
        dev = DEVICE_MAP.get(device_txt)
        if base_name and dev:
            return f"{base_name}_{dev}"
        if base_name:
            return base_name

    base_name = detect_base_name(raw_for_match)
    if base_name:
        return base_name

    safe = re.sub(r"[^a-z0-9]+", "_", raw_for_match).strip("_")
    return f"unmapped__{safe}"


VALUE_MAPS = {
    "q06_heard_e_waste": {"igen": "yes", "da": "yes", "yes": "yes", "nem": "no", "nu": "no", "no": "no"},
    "q07_joined_programs": {"igen": "yes", "da": "yes", "yes": "yes", "nem": "no", "nu": "no", "no": "no"},
    "q08_would_join_programs": {"igen": "yes", "da": "yes", "yes": "yes", "nem": "no", "nu": "no", "no": "no"},
}


def normalize_values(df: pd.DataFrame) -> pd.DataFrame:
    for col, mapping in VALUE_MAPS.items():
        if col in df.columns:
            key_map = {norm_key(k): v for k, v in mapping.items()}
            normalized = df[col].astype("string").map(lambda x: norm_key(x) if pd.notna(x) else x)
            df[col] = normalized.map(key_map).fillna(df[col])
    return df


def clean_dataset(df: pd.DataFrame, country: str) -> pd.DataFrame:
    rename = {c: canonical_col(c) for c in df.columns}
    out = df.rename(columns=rename)

    # Merge duplicate canonical columns by first non-null value.
    if out.columns.duplicated().any():
        merged = {}
        for c in dict.fromkeys(out.columns):
            same = out.loc[:, out.columns == c]
            merged[c] = same.bfill(axis=1).iloc[:, 0]
        out = pd.DataFrame(merged)

    out = normalize_values(out)
    out["country"] = country
    return out

In [ ]:
def show_unmapped(df: pd.DataFrame, title: str = "") -> None:
    mapped = {c: canonical_col(c) for c in df.columns}
    unmapped = {k: v for k, v in mapped.items() if v.startswith("unmapped__")}
    print(f"\n{title} -> Unmapped count: {len(unmapped)}")
    for i, (orig, canon) in enumerate(unmapped.items(), 1):
        print(f"{i:02d}. {orig}\n    -> {canon}")


show_unmapped(df_hu, "HU")
show_unmapped(df_ro, "RO")
show_unmapped(df_en_forms, "EN forms")
show_unmapped(df_en_swap, "EN swap")


HU -> Unmapped count: 1
01. 5. Mi ösztönözne a leginkább arra, hogy inkább megjavjíttasd a terméket csere helyett? (maximum 2 válasz bejelölhető)
    -> unmapped__5_mi_osztonozne_a_leginkabb_arra_hogy_inkabb_megjavjittasd_a_termeket_csere_helyett_maximum_2_valasz_bejelolheto

RO -> Unmapped count: 0

EN forms -> Unmapped count: 6
01. 5. To what extent are you willing to continue using an electronic device after it has been fully repaired?  [Small appliances]
    -> unmapped__5_to_what_extent_are_you_willing_to_continue_using_an_electronic_device_after_it_has_been_fully_repaired_small_appliances
02. 5. To what extent are you willing to continue using an electronic device after it has been fully repaired?  [Personal IT devices]
    -> unmapped__5_to_what_extent_are_you_willing_to_continue_using_an_electronic_device_after_it_has_been_fully_repaired_personal_it_devices
03. 5. To what extent are you willing to continue using an electronic device after it has been fully repaired?  [Large ho

In [26]:
master_df = master_df_merged
master_df.to_csv("../data/cleaned/master_dataset_v2.csv", index=False, encoding="utf-8-sig")
print(master_df.shape)


(188, 75)


In [28]:
# === UNMAPPED MERGE WORKBENCH ===
# Cél: az unmapped oszlopok kézi összekötése (HU/RO/EN variánsok egy canonical név alá)

from collections import defaultdict


def list_unmapped_columns(df: pd.DataFrame):
    return [c for c in df.columns if str(c).startswith("unmapped__")]


def base_for_grouping(col_name: str) -> str:
    """
    Egyszerű csoportosító kulcs:
    - eltávolítja a device suffixeket (_small/_personal_it/_large_household)
    - így látszik, mi tartozik logikailag egy kérdéshez
    """
    c = str(col_name)
    c = re.sub(r"_(small|personal_it|large_household)$", "", c)
    return c


def show_unmapped_groups(df: pd.DataFrame):
    um = list_unmapped_columns(df)
    groups = defaultdict(list)
    for c in um:
        groups[base_for_grouping(c)].append(c)

    print(f"Total unmapped columns: {len(um)}")
    print(f"Suggested groups: {len(groups)}\n")

    for i, (k, cols) in enumerate(sorted(groups.items(), key=lambda x: (-len(x[1]), x[0])), 1):
        print(f"{i:02d}. GROUP KEY: {k}")
        for col in sorted(cols):
            print(f"    - {col}")
        print()


# 1) Nézd meg, milyen unmapped-ek vannak és hogyan lehet őket összekötni
show_unmapped_groups(master_df)


# 2) IDE írd a kézi összekötéseket:
#    kulcs = meglévő unmapped oszlop
#    érték = cél canonical oszlopnév
manual_merge_map = {
    "unmapped__5_mi_osztonozne_a_leginkabb_arra_hogy_inkabb_megjavjittasd_a_termeket_csere_helyett_maximum_2_valasz_bejelolheto":"q20_repair_motivation",
    "unmapped__5_to_what_extent_are_you_willing_to_continue_using_an_electronic_device_after_it_has_been_fully_repaired_large_household_appliances":"q22_willing_after_full_repair_large_household",
    "unmapped__5_to_what_extent_are_you_willing_to_continue_using_an_electronic_device_after_it_has_been_fully_repaired_personal_it_devices":"q22_willing_after_full_repair_personal_it",
    "unmapped__5_to_what_extent_are_you_willing_to_continue_using_an_electronic_device_after_it_has_been_fully_repaired_small_appliances":"q22_willing_after_full_repair_small",
    "unmapped__6_to_what_extent_are_you_willing_to_continue_using_an_electronic_device_that_has_been_repaired_but_has_some_minor_issues_remained_e_g_slow_performance_scratched_or_dented_exterior_etc_large_household_appliances":"q23_willing_after_imperfect_repair_large_household",
    "unmapped__6_to_what_extent_are_you_willing_to_continue_using_an_electronic_device_that_has_been_repaired_but_has_some_minor_issues_remained_e_g_slow_performance_scratched_or_dented_exterior_etc_personal_it_devices":"q23_willing_after_imperfect_repair_personal_it",
    "unmapped__6_to_what_extent_are_you_willing_to_continue_using_an_electronic_device_that_has_been_repaired_but_has_some_minor_issues_remained_e_g_slow_performance_scratched_or_dented_exterior_etc_small_appliances":"q23_willing_after_imperfect_repair_small",
}


def apply_manual_merge_map(df: pd.DataFrame, merge_map: dict) -> pd.DataFrame:
    out = df.copy()

    # Csak létező oszlopokat vegyen figyelembe
    merge_map = {k: v for k, v in merge_map.items() if k in out.columns}

    # 1) Átnevezés
    out = out.rename(columns=merge_map)

    # 2) Ha emiatt duplikált canonical oszlop keletkezik, összeolvasztjuk first-non-null logikával
    if out.columns.duplicated().any():
        merged = {}
        for c in dict.fromkeys(out.columns):
            same = out.loc[:, out.columns == c]
            merged[c] = same.bfill(axis=1).iloc[:, 0]
        out = pd.DataFrame(merged)

    return out


# 3) Futtasd ezt, miután kitöltötted a manual_merge_map-et
master_df_merged = apply_manual_merge_map(master_df, manual_merge_map)

print("Before manual merge - unmapped:", sum(col.startswith("unmapped__") for col in master_df.columns))
print("After manual merge  - unmapped:", sum(col.startswith("unmapped__") for col in master_df_merged.columns))

# Ha oké az eredmény, ezt használd tovább:
master_df = master_df_merged

Total unmapped columns: 9
Suggested groups: 9

01. GROUP KEY: unmapped__5_mi_osztonozne_a_leginkabb_arra_hogy_inkabb_megjavjittasd_a_termeket_csere_helyett_maximum_2_valasz_bejelolheto
    - unmapped__5_mi_osztonozne_a_leginkabb_arra_hogy_inkabb_megjavjittasd_a_termeket_csere_helyett_maximum_2_valasz_bejelolheto

02. GROUP KEY: unmapped__5_to_what_extent_are_you_willing_to_continue_using_an_electronic_device_after_it_has_been_fully_repaired
    - unmapped__5_to_what_extent_are_you_willing_to_continue_using_an_electronic_device_after_it_has_been_fully_repaired

03. GROUP KEY: unmapped__5_to_what_extent_are_you_willing_to_continue_using_an_electronic_device_after_it_has_been_fully_repaired_large_household_appliances
    - unmapped__5_to_what_extent_are_you_willing_to_continue_using_an_electronic_device_after_it_has_been_fully_repaired_large_household_appliances

04. GROUP KEY: unmapped__5_to_what_extent_are_you_willing_to_continue_using_an_electronic_device_after_it_has_been_fully_repair

In [33]:
master_df.to_csv("../data/cleaned/master_dataset_final.csv", index=False, encoding="utf-8-sig")

In [34]:
import os
print(os.path.abspath("../data/cleaned/master_dataset_final.csv"))
print(master_df.shape)
print("Remaining unmapped:", sum(c.startswith("unmapped__") for c in master_df.columns))

c:\Users\fbogl\OneDrive\Dokumentumok\Egyetem\Államizsga\Allamvizsga_adatvizualizacio\data\cleaned\master_dataset_final.csv
(188, 66)
Remaining unmapped: 0
